

```
Practical Ingestion Flow:

---
VCBS Report API
      ↓
Pagination via meta / links
      ↓
Filter requiredLogin = false
      ↓
Build PDF URL
      ↓
Download PDF
      ↓
pdf-inspector
      ↓
Normalize text + metadata
      ↓
Chunk
      ↓
Embedding
      ↓
Vector / Hybrid Index
      ↓
Retriever
      ↓
LLM
---

VCBS API → paginate → filter → clean HTML → chunk → embed → index
              ↓
    for each report:
    ├─ requiredLogin = true  → skip (just store metadata)
    └─ requiredLogin = false → ingest:
         ├─ use description (HTML → clean text)
         ├─ optionally download PDF → parse with pdf-inspector
         └─ normalize into a document with metadata
        

```



In [21]:
import requests
from google.colab import drive
drive.mount('/content/drive')

import os, json

BASE_DIR = "/content/drive/MyDrive/RAG_Project"
os.makedirs(BASE_DIR, exist_ok=True)


CACHE_FILE = f"{BASE_DIR}/vcbs_reports_raw.json"
OUTPUT_FILE = f"{BASE_DIR}/vcbs_reports.json"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
url = "https://www.vcbs.com.vn/api/v1/ttpt-reports"
params = {"limit": 15, "page": 1, "locale": "en"}
headers = {"Accept": "application/json"}
REPORTS_CACHE = OUTPUT_FILE

response = requests.get(url, params=params)
response.raise_for_status()
data = response.json()

print(data.keys())
print(data["meta"])
print(data["data"][0].keys())
print(data["data"][0]["requiredLogin"])
print(data["data"][0]["file"])
print(data["meta"]["total"])

totalPages = data["meta"]["totalPages"]
totalReports = data["meta"]["total"]

dict_keys(['data', 'links', 'meta'])
{'totalPages': 276, 'limit': 15, 'total': 4136, 'page': 1}
dict_keys(['id', 'stockSymbol', 'style', 'type', 'category', 'reportYear', 'requiredLogin', 'viewedCount', 'name', 'file', 'description', 'filePages', 'createdAt', 'updatedAt', 'publishedAt'])
False
{'name': 'vcbs-26-8-2026-breaking-past-the-1800-mark-the-vn-index-surges-by-30-points.pdf'}
4136


In [23]:
def get_latest_reports(url, locale, limit=600):
    """Fetch only enough pages to get `limit` reports."""
    reports = []
    pages_needed = (limit // 15) + 1

    for page in range(1, pages_needed + 1):
        res = requests.get(url, params={"limit": 15, "page": page, "locale": locale})
        res.raise_for_status()
        batch = res.json()["data"]
        for r in batch:
            r["locale"] = locale
        reports.extend(batch)

        if page % 20 == 0:
            print(f"[{locale}] Fetched page {page}/{pages_needed}")

    print(f"[{locale}] Got {len(reports)} reports")
    return reports

In [24]:
all_reports_list = get_latest_reports(url, "en", 800)
all_reports_list.extend(get_latest_reports(url, "vi", 1200))
print(f"Fetched: {len(all_reports_list)}")

[en] Fetched page 20/54
[en] Fetched page 40/54
[en] Got 810 reports
[vi] Fetched page 20/81
[vi] Fetched page 40/81
[vi] Fetched page 60/81
[vi] Fetched page 80/81
[vi] Got 1215 reports
Fetched: 2025


In [25]:
# Filter for only requiredLogin = False using the collected list
public_reports = [r for r in all_reports_list if r.get("requiredLogin") == False]

print(f"Found {len(public_reports)} public reports.")
for report in public_reports[:10]: # Print first 10 as a sample
    print(report["file"])

Found 1863 public reports.
{'name': 'vcbs-26-8-2026-breaking-past-the-1800-mark-the-vn-index-surges-by-30-points.pdf'}
{'name': 'vcbs-25-8-2026-fluctuating-at-the-1800-point-vn-index-edges-up-by-nearly-3-points.pdf'}
{'name': 'fixed-income-report-08-17-08-21-2026-1.pdf'}
{'name': 'vcbs-24-8-2026-continuing-the-upward-trajectory-vn-index-rose-by-nearly-21-points-approaching-the-1800-level.pdf'}
{'name': 'vcbs-21-8-2026-investment-strategy-24-8-28-8-2026.pdf'}
{'name': 'vcbs-20-8-2026-stabilizing-around-the-17301740-range-the-vn-index-gains-nearly-8-points.pdf'}
{'name': 'vcbs-19-8-2026-continued-fluctuation-within-the-1720-1740-range-vn-index-drops-slightly-by-5-points.pdf'}
{'name': 'vcbs-18-8-2026-struggle-at-the-1750-resistance-level-vn-index-edges-up-by-nearly-5-points.pdf'}
{'name': 'fixed-income-report-08-10-08-14-2026-1.pdf'}
{'name': 'vcbs-17-8-2026-hesitating-around-the-1720-mark-vn-index-edged-down-by-nearly-2-points.pdf'}


In [26]:
import json
import os

In [27]:
def load_cached_reports():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return None

In [28]:
def save_reports_cache(reports):
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(reports, f, ensure_ascii=False)
    print(f"Cached {len(reports)} reports to {CACHE_FILE}")

In [29]:
def has_new_data(url):
    """Quick check — hit page 1 of both locales, compare total against cache."""
    remote_total = 0
    for locale in ["en", "vi"]:
        res = requests.get(url, params={"limit": 1, "page": 1, "locale": locale})
        res.raise_for_status()
        remote_total += res.json()["meta"]["total"]

    cached = load_cached_reports()
    if cached is None:
        print(f"No cache found. Remote has {remote_total} reports.")
        return True

    print(f"Cache: {len(cached)} reports. Remote: {remote_total} reports.")
    return remote_total != len(cached)

In [30]:
# Load existing IDs from saved file
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        existing = json.load(f)
    existing_ids = {r["id"] for r in existing}
    print(f"Existing: {len(existing_ids)} reports")
else:
    existing_ids = set()
    print("No existing file — full fetch")

# Fetch only new
if existing_ids:
    new_reports = get_latest_reports(url, "en", 100)
    new_reports.extend(get_latest_reports(url, "vi", 100))
    new_reports = [r for r in new_reports if r["id"] not in existing_ids]
else:
    new_reports = get_latest_reports(url, "en", 800)
    new_reports.extend(get_latest_reports(url, "vi", 1200))

print(f"New reports fetched: {len(new_reports)}")

Existing: 1000 reports
[en] Got 105 reports
[vi] Got 105 reports
New reports fetched: 117


In [31]:
def deduplicate(reports):
    """Deduplicate by report ID. English versions take priority."""
    seen = {}
    for r in reports:
        rid = r["id"]
        if rid not in seen:
            seen[rid] = r
        else:
            # if current is English and existing isn't, replace
            if r.get("locale") == "en" or "locale" not in seen[rid]:
                seen[rid] = r
    print(f"Before dedup: {len(reports)} → After: {len(seen)}")
    return list(seen.values())

In [32]:
import re
import html

def clean_html(html_text):
    if not html_text:
        return ""
    text = re.sub(r'<[^>]+>', '', html_text)
    text = html.unescape(text)           # decode &nbsp; &#39; etc.
    return text.strip()

In [33]:
def normalize_report(report):
    published = report.get("publishedAt", "")
    date_part = published[:10].replace("-", "") if published else ""
    file_obj = report.get("file") or {}
    file_name = file_obj.get("name", "")
    pdf_url = f"https://www.vcbs.com.vn/storage/ttpt_reports/{date_part}/{file_name}" if date_part and file_name else ""

    return {
        "id": report.get("id"),
        "name": report.get("name", ""),
        "stockSymbol": report.get("stockSymbol", ""),
        "category": report.get("category", ""),
        "description": clean_html(report.get("description", "")),
        "publishedAt": published,
        "pdf_url": pdf_url,
        "locale": report.get("locale", ""),
    }

In [34]:
# Filter public
public_reports = [r for r in all_reports_list if not r.get("requiredLogin")]
print(f"Public: {len(public_reports)}")

# Deduplicate
unique_reports = deduplicate(public_reports)

# Normalize
normalized = [normalize_report(r) for r in unique_reports]
print(f"Normalized: {len(normalized)}")

# Latest 1000
normalized.sort(key=lambda r: r["publishedAt"] or "", reverse=True)
latest = normalized[:1000]
print(f"Latest 1000: {latest[-1]['publishedAt'][:10]} → {latest[0]['publishedAt'][:10]}")

# Save
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(latest, f, ensure_ascii=False, indent=2)
print(f"Saved {len(latest)} reports to vcbs_reports.json")

Public: 1863
Before dedup: 1863 → After: 1851
Normalized: 1851
Latest 1000: 2025-06-25 → 2026-08-26
Saved 1000 reports to vcbs_reports.json


In [35]:
import requests

res = requests.get("https://www.vcbs.com.vn/api/v1/ttpt-reports", params={
    "locale": "en",
    "page": 1,
    "per_page": 5
})
data = res.json()
for r in data["data"][:5]:
    print(r.get("publishedAt", "?")[:10], r.get("name", "?")[:60])

2026-08-26 VCBS_26.8.2026 Breaking past the 1,800 mark, the VN-Index su
2026-08-25 VCBS_25.8.2026 Fluctuating at the 1,800-point, VN-Index edge
2026-08-24 Fixed-income report  08 17 - 08 21 2026
2026-08-24 VCBS_24.8.2026 Continuing the upward trajectory, VN-Index ro
2026-08-21 VCBS_21.8.2026 Investment strategy 24.8 - 28.8.2026
